In [8]:
# [Cell 1] 环境配置、物理拓扑与全局控制 (Jupyter 独立运行基石)
# ==============================================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
import random
import warnings

warnings.filterwarnings("ignore")

# ------------------------------------------
# 🔒 1. 核心控制台：锁定所有随机性
# ------------------------------------------
def set_seed(seed=42):
    """锁定 PyTorch、Numpy 和 Python 所有的底层骰子"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def get_obs_indices(num_nodes=33, seed=42):
    """动态但固定地生成 15% 观测点 (永远包含 0 号平衡节点)"""
    random.seed(seed)  # 局部锁死
    num_obs = int(num_nodes * 0.15)  
    all_idx = list(range(1, num_nodes))
    return [0] + sorted(random.sample(all_idx, num_obs - 1))


# 初始化全局状态
set_seed(42)
global_obs_indices = get_obs_indices(33, 42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 炼丹炉就绪! 算力: {device}")
print(f"📡 锁定的 15% PMU 观测点: {global_obs_indices}")

# ------------------------------------------
# ⚡ 2. 提取 33 节点导纳矩阵 Ybus
# ------------------------------------------
branch_data = np.array([
    [1, 2, 0.00057, 0.00029], [2, 3, 0.00307, 0.00156], [3, 4, 0.00228, 0.00117],
    [4, 5, 0.00237, 0.00121], [5, 6, 0.00511, 0.00441], [6, 7, 0.00116, 0.00336],
    [7, 8, 0.00443, 0.00146], [8, 9, 0.00642, 0.00461], [9, 10, 0.00651, 0.00461],
    [10, 11, 0.00122, 0.00040], [11, 12, 0.00233, 0.00074], [12, 13, 0.00915, 0.00720],
    [13, 14, 0.00337, 0.00444], [14, 15, 0.00368, 0.00328], [15, 16, 0.00465, 0.00340],
    [16, 17, 0.00804, 0.01073], [17, 18, 0.00456, 0.00358], [2, 19, 0.00102, 0.00097],
    [19, 20, 0.00938, 0.00845], [20, 21, 0.00255, 0.00298], [21, 22, 0.00442, 0.00584],
    [3, 23, 0.00281, 0.00192], [23, 24, 0.00560, 0.00442], [24, 25, 0.00559, 0.00437],
    [6, 26, 0.00126, 0.00064], [26, 27, 0.00177, 0.00090], [27, 28, 0.00660, 0.00582],
    [28, 29, 0.00501, 0.00437], [29, 30, 0.00316, 0.00161], [30, 31, 0.00608, 0.00600],
    [31, 32, 0.00193, 0.00225], [32, 33, 0.00212, 0.00330]
])

num_nodes = 33
y_bus = np.zeros((num_nodes, num_nodes), dtype=complex)
for i in range(len(branch_data)):
    f, t = int(branch_data[i, 0] - 1), int(branch_data[i, 1] - 1)
    y_line = 1 / (branch_data[i, 2] + 1j * branch_data[i, 3])
    y_bus[f, f] += y_line
    y_bus[t, t] += y_line
    y_bus[f, t] = y_bus[t, f] = -y_line

G_tensor = torch.tensor(y_bus.real, dtype=torch.float32).to(device)
B_tensor = torch.tensor(y_bus.imag, dtype=torch.float32).to(device)


# ==============================================================================
# [Cell 2] 核心武器库：物理方程、网络架构与观测控制 (Jupyter 独立版)
# ==============================================================================

# ------------------------------------------
# 1. 盲区遮蔽引擎：模拟只有 15% 传感器的数据流
# ------------------------------------------
def apply_blind_zone(batch_x, obs_indices, mean_t, scale_t):
    physical_zero_std = (0.0 - mean_t) / scale_t
    masked_x = physical_zero_std.repeat(batch_x.shape[0], 1)
    for idx in obs_indices:
        masked_x[:, idx] = batch_x[:, idx]  
        masked_x[:, idx + 33] = batch_x[:, idx + 33]  
    return masked_x


# ------------------------------------------
# 2. 潮流方程引擎：PyTorch 矩阵化计算
# ------------------------------------------
def calculate_physics_p_torch(V_pred, theta_pred, G_t, B_t):
    delta_theta = theta_pred.unsqueeze(2) - theta_pred.unsqueeze(1)
    cos_matrix = torch.cos(delta_theta)
    sin_matrix = torch.sin(delta_theta)

    p_term = G_t * cos_matrix + B_t * sin_matrix
    q_term = G_t * sin_matrix - B_t * cos_matrix

    sum_p = torch.sum(V_pred.unsqueeze(1) * p_term, dim=2)
    sum_q = torch.sum(V_pred.unsqueeze(1) * q_term, dim=2)

    P_calc = V_pred * sum_p
    Q_calc = V_pred * sum_q
    return P_calc, Q_calc


# ------------------------------------------
# 3. R-PINN 网络架构：带 Asymmetric Residual Scaling (ARS)
# ------------------------------------------
class PowerGridPINN(nn.Module):
    def __init__(self, input_dim=66):
        super(PowerGridPINN, self).__init__()
        self.hidden_layers = nn.Sequential(
            nn.Linear(input_dim, 256), nn.SiLU(),
            nn.Linear(256, 256), nn.SiLU(),
            nn.Linear(256, 256)
        )
        self.output_layers = nn.Linear(256, 66)

    def forward(self, x):
        features = self.hidden_layers(x)
        output = self.output_layers(features)

        # 🌟 核心 ARS 机制修改：将相角搜索空间从 0.01 收窄至 0.005
        Vm_pred = output[:, :33] * 0.1 + 1.0
        theta_pred = output[:, 33:] * 0.005 + 0.0

        Vm_pred[:, 0] = 1.0
        theta_pred[:, 0] = 0.0
        return Vm_pred, theta_pred


# ------------------------------------------
# 4. 联合 Loss 函数：数据锚定 + 物理一致性 + 电压墙 (修正 Slack 掩码)
# ------------------------------------------
class PowerPINNLoss(nn.Module):
    def __init__(self, G, B, obs_idx):
        super(PowerPINNLoss, self).__init__()
        self.G, self.B, self.obs_idx = G, B, obs_idx
        self.mse_tool = nn.MSELoss()

    def forward(self, V_pred, theta_pred, P_real_target, Q_real_target, V_real, T_real, p_weight, obs_weight=500000):
        # 1. 潮流物理残差 (KCL)
        P_calc, Q_calc = calculate_physics_p_torch(V_pred, theta_pred, self.G, self.B)
        
        # 33节点只约束 1-32 号节点的功率平衡。0 号节点作为平衡节点，不参与功率守恒约束。
        P_loss = self.mse_tool(P_calc[:, 1:], P_real_target[:, 1:])
        Q_loss = self.mse_tool(Q_calc[:, 1:], Q_real_target[:, 1:])

        # 2. 观测点数据偏差 (15% 传感器的锚定)
        # 🌟 核心修改：保持 Mean-Loss (self.mse_tool)，但为相角赋予 1000 倍独立权重
        obs_loss_v = self.mse_tool(V_pred[:, self.obs_idx], V_real[:, self.obs_idx])
        obs_loss_t = self.mse_tool(theta_pred[:, self.obs_idx], T_real[:, self.obs_idx])
        
        # 将相角观测损失放大 1000 倍，暴力解决梯度饥饿问题
        obs_loss = obs_loss_v + 1000.0 * obs_loss_t

        # 3. 边界罚项 (防止电压飞了)
        penalty_low = torch.nn.functional.relu(0.85 - V_pred)
        penalty_high = torch.nn.functional.relu(V_pred - 1.1)

        return (
                p_weight * (P_loss + Q_loss) +
                1000 * torch.mean(penalty_low + penalty_high) +
                obs_weight * obs_loss
        )
# ------------------------------------------
# 5. 数据装载器
# ------------------------------------------
class MyDataset(Dataset):
    def __init__(self, features, labels):
        self.features = features
        self.labels = labels

    def __len__(self): return len(self.labels)

    def __getitem__(self, idx): return self.features[idx], self.labels[idx]


print("✅ Cell 2 [Mean-Loss + ARS 强化] 加载完毕！物理掩码、网络架构和独立种子机制已全部就位。")

# ==============================================================================
# [Cell 3] 数据引擎：脱水、归一化与全链条种子锁定 (升级 132 维目标)
# ==============================================================================

# 🚀 绝杀操作：在加载数据前再次重置种子，确保 DataLoader 的 shuffle 逻辑被物理锁死
set_seed(42)

print("📥 正在从本地磁盘读取 50,000 条 33 节点物理样本...")
# 注意：路径请保持你电脑上的原始路径
df = pd.read_csv(r'IEEE33_50000_Safe_Final.csv')
raw_data = df.values
m = raw_data.shape[0]

# 1. 物理维度重塑 [样本数, 节点数, 特征数(P, Q, V, Theta)]
data_3d = raw_data.reshape(m, 33, 4)

# 2. 物理脱水：兆瓦(MW/MVar) -> 标幺值(p.u.) 
P_real_pu = data_3d[:, :, 0].copy() / 100.0  
Q_real_pu = data_3d[:, :, 1].copy() / 100.0
V_real_pu = data_3d[:, :, 2].copy()  
T_real_rad = np.deg2rad(data_3d[:, :, 3].copy()) # 💡 提取真实的 Theta (弧度)

# 3. 输入特征归一化 (StandardScaler)
X_raw = data_3d[:, :, 0:2].reshape(m, 66)  
ssl = StandardScaler()
X_norm = ssl.fit_transform(X_raw)

# 4. 提取归一化参数
mean_tensor = torch.tensor(ssl.mean_, dtype=torch.float32).to(device)
scale_tensor = torch.tensor(ssl.scale_, dtype=torch.float32).to(device)

# 5. 目标值拼装升级：[P, Q, V, Theta] -> 总共 132 维！
PQVT_target_up = np.concatenate([P_real_pu, Q_real_pu, V_real_pu, T_real_rad], axis=1)

# 6. 转化为 GPU 计算张量
X_tensor = torch.tensor(X_norm, dtype=torch.float32).to(device)
Target_tensor = torch.tensor(PQVT_target_up, dtype=torch.float32).to(device)

# 7. 创建数据集与装载器
g = torch.Generator()
g.manual_seed(42)

train_dataset = MyDataset(X_tensor, Target_tensor)
train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True,  
    generator=g  
)

print("-" * 50)
print(f"✅ 数据引擎装载完毕！")
print(f"📊 样本总数: {m} | 输入维度: 66 | 目标维度: 132")
print(f"🔒 数据洗牌顺序已锁定，实验可复现性：100%")
print("-" * 50)

🚀 炼丹炉就绪! 算力: cuda
📡 锁定的 15% PMU 观测点: [0, 2, 8, 18]
✅ Cell 2 [Mean-Loss + ARS 强化] 加载完毕！物理掩码、网络架构和独立种子机制已全部就位。
📥 正在从本地磁盘读取 50,000 条 33 节点物理样本...
--------------------------------------------------
✅ 数据引擎装载完毕！
📊 样本总数: 50000 | 输入维度: 66 | 目标维度: 132
🔒 数据洗牌顺序已锁定，实验可复现性：100%
--------------------------------------------------


In [2]:
# ==============================================================================
# 🔬 [敏感性分析] 33 节点 R-PINN：固定其他条件，仅改变观测节点位置，重复10次
# ==============================================================================
import time
import copy

n_runs = 10
results = []

print("\n" + "="*70)
print("🔬 启动观测节点敏感性分析 (10 次独立训练，仅改变 15% 传感器位置)")
print("="*70)

base_obs_seeds = [100, 200, 300, 400, 500, 600, 700, 800, 900, 1000]  # 控制观测节点选择的种子

for run_idx, obs_seed in enumerate(base_obs_seeds):
    print(f"\n{'='*50}")
    print(f"▶ 第 {run_idx+1}/{n_runs} 次训练，观测节点种子: {obs_seed}")
    print(f"{'='*50}")
    
    # ---------- 每次训练前锁定全局种子（保证模型初始化、数据顺序完全一致） ----------
    set_seed(42)
    
    # ---------- 根据当前种子生成新的观测节点列表 ----------
    current_obs = get_obs_indices(33, obs_seed)
    print(f"📡 本次观测节点: {current_obs}")
    
    # ---------- 重新构建 DataLoader（数据顺序因 set_seed(42) 与之前完全一致） ----------
    train_dataset = MyDataset(X_tensor, Target_tensor)
    g = torch.Generator()
    g.manual_seed(42)
    train_loader = DataLoader(
        train_dataset,
        batch_size=128,
        shuffle=True,
        generator=g
    )
    
    # ---------- 初始化模型（权重受 set_seed(42) 控制，每次相同） ----------
    model = PowerGridPINN(input_dim=66).to(device)
    with torch.no_grad():
        model.output_layers.bias[:33].fill_(-0.5)
    
    pinn_loss = PowerPINNLoss(G_tensor, B_tensor, current_obs)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=40, gamma=0.5)
    
    # ---------- 训练（超参数与主实验完全一致） ----------
    for epoch in range(150):
        p_weight = 1000 if epoch < 50 else 5000
        model.train()
        for batch_x, batch_targets in train_loader:
            optimizer.zero_grad()
            real_p, real_q = batch_targets[:, :33], batch_targets[:, 33:66]
            real_v, real_t = batch_targets[:, 66:99], batch_targets[:, 99:132]
            
            mask_x = apply_blind_zone(batch_x, current_obs, mean_tensor, scale_tensor)
            V_g, T_g = model(mask_x)
            
            loss = pinn_loss(V_g, T_g, real_p, real_q, real_v, real_t, p_weight, obs_weight=2500000)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        scheduler.step()
    
    # ---------- 评估 ----------
    model.eval()
    with torch.no_grad():
        p_Vm, p_Tm = model(apply_blind_zone(X_tensor[:1000], current_obs, mean_tensor, scale_tensor))
        r_Vm = V_real_pu[:1000]   # 注意：这里 data_3d 在 Cell 3 中定义，是全局变量
        r_Tm = T_real_rad[:1000]
        
        mae_v = np.mean(np.abs(p_Vm.cpu().numpy() - r_Vm))
        rmse_v = np.sqrt(np.mean((p_Vm.cpu().numpy() - r_Vm)**2))
        mae_t = np.mean(np.abs(p_Tm.cpu().numpy() - r_Tm))
        rmse_t = np.sqrt(np.mean((p_Tm.cpu().numpy() - r_Tm)**2))
    
    results.append({
        "run": run_idx+1,
        "obs_seed": obs_seed,
        "obs_nodes": current_obs,
        "V_MAE": mae_v,
        "V_RMSE": rmse_v,
        "T_MAE": mae_t,
        "T_RMSE": rmse_t
    })
    print(f"✅ 第 {run_idx+1} 次完成: V_MAE={mae_v:.6e}, T_MAE={mae_t:.6e}")

# ---------- 最终汇总 ----------
print("\n" + "="*70)
print("🔬 敏感性分析结果汇总 (10 次随机观测节点)")
print("="*70)
print(f"{'Run':<5} {'Obs Seed':<10} {'V_MAE':<15} {'V_RMSE':<15} {'T_MAE':<15} {'T_RMSE':<15}")
print("-"*70)
for r in results:
    print(f"{r['run']:<5} {r['obs_seed']:<10} {r['V_MAE']:<15.6e} {r['V_RMSE']:<15.6e} {r['T_MAE']:<15.6e} {r['T_RMSE']:<15.6e}")

# 计算统计量
v_mae_arr = np.array([r['V_MAE'] for r in results])
v_rmse_arr = np.array([r['V_RMSE'] for r in results])
t_mae_arr = np.array([r['T_MAE'] for r in results])
t_rmse_arr = np.array([r['T_RMSE'] for r in results])

print("\n📊 统计量:")
print(f"  Voltage MAE  : mean={np.mean(v_mae_arr):.6e}, std={np.std(v_mae_arr):.6e}")
print(f"  Voltage RMSE : mean={np.mean(v_rmse_arr):.6e}, std={np.std(v_rmse_arr):.6e}")
print(f"  Phase MAE    : mean={np.mean(t_mae_arr):.6e}, std={np.std(t_mae_arr):.6e}")
print(f"  Phase RMSE   : mean={np.mean(t_rmse_arr):.6e}, std={np.std(t_rmse_arr):.6e}")


🔬 启动观测节点敏感性分析 (10 次独立训练，仅改变 15% 传感器位置)

▶ 第 1/10 次训练，观测节点种子: 100
📡 本次观测节点: [0, 10, 12, 30]
✅ 第 1 次完成: V_MAE=3.900755e-03, T_MAE=1.004812e-03

▶ 第 2/10 次训练，观测节点种子: 200
📡 本次观测节点: [0, 2, 3, 14]
✅ 第 2 次完成: V_MAE=4.717395e-03, T_MAE=1.403179e-03

▶ 第 3/10 次训练，观测节点种子: 300
📡 本次观测节点: [0, 1, 23, 32]
✅ 第 3 次完成: V_MAE=7.589401e-03, T_MAE=1.218893e-03

▶ 第 4/10 次训练，观测节点种子: 400
📡 本次观测节点: [0, 6, 17, 20]
✅ 第 4 次完成: V_MAE=4.535256e-03, T_MAE=1.816560e-03

▶ 第 5/10 次训练，观测节点种子: 500
📡 本次观测节点: [0, 17, 25, 30]
✅ 第 5 次完成: V_MAE=2.959322e-03, T_MAE=6.656442e-04

▶ 第 6/10 次训练，观测节点种子: 600
📡 本次观测节点: [0, 17, 23, 29]
✅ 第 6 次完成: V_MAE=2.036614e-03, T_MAE=9.882026e-04

▶ 第 7/10 次训练，观测节点种子: 700
📡 本次观测节点: [0, 11, 13, 29]
✅ 第 7 次完成: V_MAE=3.506322e-03, T_MAE=1.178181e-03

▶ 第 8/10 次训练，观测节点种子: 800
📡 本次观测节点: [0, 4, 31, 32]
✅ 第 8 次完成: V_MAE=6.996287e-03, T_MAE=1.113507e-03

▶ 第 9/10 次训练，观测节点种子: 900
📡 本次观测节点: [0, 7, 15, 24]
✅ 第 9 次完成: V_MAE=3.722943e-03, T_MAE=1.972201e-03

▶ 第 10/10 次训练，观测节点种子: 1000
📡 本次观测节点: [0, 7, 26,

In [9]:
# ==============================================================================
# 🔬 [敏感性分析 - 学习率调度器鲁棒性] 33节点 R-PINN
# 固定观测节点，改变 StepLR 的 step_size 和 gamma，每种组合跑一次，共10组
# ==============================================================================
import time
import itertools

# 定义多组学习率调度超参数
step_sizes = [30, 35, 40, 45, 50]
gammas = [0.3, 0.4, 0.5, 0.6, 0.7]
# 从所有组合中随机选取10组，保证多样性
np.random.seed(42)
all_combos = list(itertools.product(step_sizes, gammas))
selected_combos = np.random.choice(len(all_combos), size=10, replace=False)
lr_schedules = [all_combos[i] for i in selected_combos]

n_runs = len(lr_schedules)
results_lr = []

# 固定观测节点（33 节点专用种子 42）
set_seed(42)
fixed_obs = get_obs_indices(33, 42)   # 比如 [0, 2, 8, 18, 24]

print("\n" + "="*70)
print("🔬 启动学习率调度器敏感性分析 (10种不同 StepLR 超参数)")
print(f"   固定观测节点: {fixed_obs}")
print("="*70)

for run_idx, (step_sz, gamma) in enumerate(lr_schedules):
    print(f"\n{'='*50}")
    print(f"▶ 第 {run_idx+1}/{n_runs} 次训练: step_size={step_sz}, gamma={gamma}")
    print(f"{'='*50}")
    
    # ---------- 锁定全局种子 ----------
    set_seed(42)
    
    # ---------- 固定观测节点 ----------
    current_obs = fixed_obs
    
    # ---------- 重构 DataLoader ----------
    train_dataset = MyDataset(X_tensor, Target_tensor)
    g = torch.Generator()
    g.manual_seed(42)
    train_loader = DataLoader(
        train_dataset,
        batch_size=128,
        shuffle=True,
        generator=g
    )
    
    # ---------- 模型初始化 ----------
    model = PowerGridPINN(input_dim=66).to(device)
    with torch.no_grad():
        model.output_layers.bias[:33].fill_(-0.5)
    
    pinn_loss = PowerPINNLoss(G_tensor, B_tensor, current_obs)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    # 🌟 核心改动点：使用不同的 StepLR 超参数
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=step_sz, gamma=gamma)
    
    # ---------- 训练（动态物理权重与主实验一致） ----------
    for epoch in range(150):
        p_weight = 1000 if epoch < 50 else 5000   # 物理权重动态调度
        
        model.train()
        for batch_x, batch_targets in train_loader:
            optimizer.zero_grad()
            real_p, real_q = batch_targets[:, :33], batch_targets[:, 33:66]
            real_v, real_t = batch_targets[:, 66:99], batch_targets[:, 99:132]
            
            mask_x = apply_blind_zone(batch_x, current_obs, mean_tensor, scale_tensor)
            V_g, T_g = model(mask_x)
            
            loss = pinn_loss(V_g, T_g, real_p, real_q, real_v, real_t, p_weight, obs_weight=2500000)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        scheduler.step()
    
    # ---------- 评估 ----------
    model.eval()
    with torch.no_grad():
        p_Vm, p_Tm = model(apply_blind_zone(X_tensor[:1000], current_obs, mean_tensor, scale_tensor))
        r_Vm = V_real_pu[:1000]
        r_Tm = T_real_rad[:1000]
        
        mae_v = np.mean(np.abs(p_Vm.cpu().numpy() - r_Vm))
        rmse_v = np.sqrt(np.mean((p_Vm.cpu().numpy() - r_Vm)**2))
        mae_t = np.mean(np.abs(p_Tm.cpu().numpy() - r_Tm))
        rmse_t = np.sqrt(np.mean((p_Tm.cpu().numpy() - r_Tm)**2))
    
    results_lr.append({
        "run": run_idx+1,
        "step_size": step_sz,
        "gamma": gamma,
        "V_MAE": mae_v,
        "V_RMSE": rmse_v,
        "T_MAE": mae_t,
        "T_RMSE": rmse_t
    })
    print(f"✅ 第 {run_idx+1} 次完成: V_MAE={mae_v:.6e}, T_MAE={mae_t:.6e}")

# ---------- 汇总结果 ----------
print("\n" + "="*70)
print("🔬 学习率调度器敏感性分析结果汇总")
print("="*70)
print(f"{'Run':<5} {'step_size':<10} {'gamma':<8} {'V_MAE':<15} {'V_RMSE':<15} {'T_MAE':<15} {'T_RMSE':<15}")
print("-"*70)
for r in results_lr:
    print(f"{r['run']:<5} {r['step_size']:<10} {r['gamma']:<8.2f} {r['V_MAE']:<15.6e} {r['V_RMSE']:<15.6e} {r['T_MAE']:<15.6e} {r['T_RMSE']:<15.6e}")

# 统计量
v_mae_arr = np.array([r['V_MAE'] for r in results_lr])
v_rmse_arr = np.array([r['V_RMSE'] for r in results_lr])
t_mae_arr = np.array([r['T_MAE'] for r in results_lr])
t_rmse_arr = np.array([r['T_RMSE'] for r in results_lr])

print("\n📊 统计量 (10种学习率调度下):")
print(f"  Voltage MAE  : mean={np.mean(v_mae_arr):.6e}, std={np.std(v_mae_arr):.6e}")
print(f"  Voltage RMSE : mean={np.mean(v_rmse_arr):.6e}, std={np.std(v_rmse_arr):.6e}")
print(f"  Phase MAE    : mean={np.mean(t_mae_arr):.6e}, std={np.std(t_mae_arr):.6e}")
print(f"  Phase RMSE   : mean={np.mean(t_rmse_arr):.6e}, std={np.std(t_rmse_arr):.6e}")


🔬 启动学习率调度器敏感性分析 (10种不同 StepLR 超参数)
   固定观测节点: [0, 2, 8, 18]

▶ 第 1/10 次训练: step_size=35, gamma=0.6
✅ 第 1 次完成: V_MAE=9.322282e-03, T_MAE=1.899151e-03

▶ 第 2/10 次训练: step_size=45, gamma=0.4
✅ 第 2 次完成: V_MAE=9.332621e-03, T_MAE=1.896859e-03

▶ 第 3/10 次训练: step_size=30, gamma=0.3
✅ 第 3 次完成: V_MAE=9.359051e-03, T_MAE=1.895238e-03

▶ 第 4/10 次训练: step_size=50, gamma=0.6
✅ 第 4 次完成: V_MAE=9.292172e-03, T_MAE=1.907476e-03

▶ 第 5/10 次训练: step_size=40, gamma=0.4
✅ 第 5 次完成: V_MAE=9.353573e-03, T_MAE=1.896175e-03

▶ 第 6/10 次训练: step_size=35, gamma=0.7
✅ 第 6 次完成: V_MAE=9.278269e-03, T_MAE=1.901959e-03

▶ 第 7/10 次训练: step_size=40, gamma=0.6
✅ 第 7 次完成: V_MAE=9.303602e-03, T_MAE=1.897779e-03

▶ 第 8/10 次训练: step_size=30, gamma=0.4
✅ 第 8 次完成: V_MAE=9.349936e-03, T_MAE=1.893424e-03

▶ 第 9/10 次训练: step_size=50, gamma=0.5
✅ 第 9 次完成: V_MAE=9.247979e-03, T_MAE=1.901127e-03

▶ 第 10/10 次训练: step_size=35, gamma=0.3
✅ 第 10 次完成: V_MAE=9.351581e-03, T_MAE=1.895302e-03

🔬 学习率调度器敏感性分析结果汇总
Run   step_size  gamma    V_